## Feature Engineering


### Imports

In [1]:
from feature_engineering import merge_and_engineer_features, load_and_clean_data

In [2]:
# Step 1: Load and Clean Data
df_energy_clean, city_dfs_clean = load_and_clean_data(
    energy_path="data/energy_dataset.csv", 
    weather_path="data/weather_features.csv"
)

# Step 2: Feature Engineering
df_model = merge_and_engineer_features(df_energy_clean, city_dfs_clean)

# Step 3: Verify Results
print("\n--- SAMPLE OUTPUT ---")
cols_to_show = [
    'total load actual', 'day_type', 'is_holiday_madrid', 'is_holiday_barcelona',
    'temp_madrid', 'temp_barcelona', 'temp_madrid_lag_24', 'load_rolling_mean_24h'
]
display(df_model[cols_to_show].head())


--- 1. LOADING AND CLEANING DATA ---
Processing Energy Dataset...
Processing Weather Dataset...
Data loading and cleaning complete.

--- 2. FEATURE ENGINEERING ---
Merging individual city weather datasets...
Creating temporal variables and cyclical encoding...
Incorporating external covariates: National and Regional Holidays...
Generating safe lags for Day-Ahead prediction...
Generating safe rolling window statistics...
Dropping rows with NaNs generated by lags (first 8 days)...
✅ Process completed. Final dimensions: (34873, 100)

--- SAMPLE OUTPUT ---


,total load actual,day_type,is_holiday_madrid,is_holiday_barcelona,temp_madrid,temp_barcelona,temp_madrid_lag_24,load_rolling_mean_24h
time,,,,,,,,
2015-01-08 22:00:00+00:00,26436.0,0,0,0,-3.856,9.6,-5.0910,32105.000000
2015-01-08 23:00:00+00:00,27485.0,0,0,0,-3.856,10.4,-5.0910,31949.583333
2015-01-09 00:00:00+00:00,25750.0,0,0,0,-3.856,10.3,-5.4325,31810.500000
2015-01-09 01:00:00+00:00,24760.0,0,0,0,-3.902,10.0,-5.7740,31679.791667
2015-01-09 02:00:00+00:00,24188.0,0,0,0,-3.902,10.4,-5.7740,31546.375000


In [ ]:
# Import the functions from your newly created .py file
from evaluation import temporal_train_test_split, evaluate_tso_baseline, plot_forecast_vs_actual

# 1. Split the data (assuming df_model is your fully engineered dataframe)
X_train, X_test, y_train, y_test = temporal_train_test_split(
    df=df_model, 
    target_col='total load actual', 
    cutoff_date='2018-01-01 00:00:00+00:00'
)

# 2. Get the baseline metrics to beat
tso_forecast, tso_metrics = evaluate_tso_baseline(
    y_test=y_test, 
    energy_csv_path="data/energy_dataset.csv"
)

# 3. Visualize the first week of 2018
plot_forecast_vs_actual(y_actual=y_test, y_pred=tso_forecast, model_name="TSO Baseline", window_hours=168)